## Import Libraries

In [ ]:
pip install cleanlab

In [ ]:
# GOOGLE DRIVE
PROJECT_DIR = "/content/drive/MyDrive/FDL Project"
DATA_DIR = f"{PROJECT_DIR}/data/flower_photos"
SPLIT_DIR = f"{PROJECT_DIR}/data/splits"
MODEL_DIR = f"{PROJECT_DIR}/models"
SCRIPT_DIR = f"{PROJECT_DIR}/scripts"
PLOT_DIR = f"{PROJECT_DIR}/plots"
HISTORY_DIR = f"{PROJECT_DIR}/history"

In [ ]:
#import cleanlab
import sys
import keras
import numpy as np
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
if SCRIPT_DIR not in sys.path:
    sys.path.append(SCRIPT_DIR)

from preprocessing import load_common_datasets

## Get Prediction Probabilities



In [ ]:
df_main = pd.read_csv(f"{SPLIT_DIR}/train_folds.csv", dtype={'image_path': str, 'fold': int})
unique_folds = sorted(df_main['fold'].unique())
print(f"Found folds: {unique_folds}. \n")

In [ ]:
pred_probs = np.zeros((len(df_main), 5), dtype=np.float32)
NUM_FOLDS = len(unique_folds)

for current_fold in unique_folds:
  print(f"========== TRAINING FOLD {current_fold + 1} / {NUM_FOLDS} ==========")

  # PREPARE TRAIN AND VAL DATA HERE

  _, val_ds, _ , _ = load_common_datasets(
    project_dir=PROJECT_DIR,
    val_fold=current_fold,
    img_size=224,
    batch_size=32,
    normalization="rescale_m1_1",
    shuffle_train = False
  )

  model = keras.models.load_model(f"{MODEL_DIR}/SCRATCH_v8.0_fold_{current_fold}_2026-08-30_15-04-30.keras")

  fold_probs = model.predict(val_ds)

  val_indices = df_main[df_main["fold"] == current_fold].index.values
  pred_probs[val_indices] = fold_probs



In [ ]:
labels = df_main["label"].values
issues = cleanlab.filter.find_label_issues(
    labels=labels,
    pred_probs=pred_probs,
    return_indices_ranked_by="self_confidence"
)

print(f"\nTotal samples: {len(df_main)}")
print(f"Flagged problematic samples: {len(issues)}")

# Create clean CSV without the flagged rows
df_clean = df_main.drop(index=issues).reset_index(drop=True)
# df_clean.to_csv("train_folds_clean.csv", index=False)

In [ ]:
df_clean.head(5)

In [ ]:
# Class: tulips          ---> Numeric Label: 0
#   Class: dandelion       ---> Numeric Label: 1
#   Class: daisy           ---> Numeric Label: 2
#   Class: sunflowers      ---> Numeric Label: 3
#   Class: roses           ---> Numeric Label: 4

In [ ]:
import math
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
from PIL import Image

class_names = {
    0: "tulips",
    1: "dandelion",
    2: "daisy",
    3: "sunflowers",
    4: "roses"
}

PAGE_SIZE = 16  # 4x4 grid per page
total_pages = math.ceil(len(issues) / PAGE_SIZE)

def show_page(page):
    start_idx = (page - 1) * PAGE_SIZE
    end_idx = min(start_idx + PAGE_SIZE, len(issues))
    current_issues = issues[start_idx:end_idx]

    fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(16, 16))
    axes = axes.flatten()

    for i in range(len(axes)):
        if i < len(current_issues):
            idx = current_issues[i]
            rank = start_idx + i + 1

            img_path = df_main.loc[idx, "image_path"]
            given_idx = df_main.loc[idx, "label"]
            given_name = class_names.get(given_idx, str(given_idx))

            probs = pred_probs[idx]
            pred_idx = probs.argmax()
            pred_name = class_names.get(pred_idx, str(pred_idx))

            try:
                img = Image.open(img_path)
                axes[i].imshow(img)
            except Exception:
                axes[i].text(0.5, 0.5, "Image Load Error", ha="center")

            axes[i].axis("off")
            axes[i].set_title(
                f"#{rank} (Row {idx})\n"
                f"Given: {given_name} ({probs[given_idx]:.1%})\n"
                f"Pred: {pred_name} ({probs[pred_idx]:.1%})",
                fontsize=9, color="darkred"
            )
        else:
            axes[i].axis("off")  # Hide unused grid tiles

    plt.tight_layout()
    plt.show()

In [ ]:
widgets.interact(
    show_page,
    page=widgets.IntSlider(
        min=1,
        max=total_pages,
        step=1,
        value=1,
        description=f"Page (1-{total_pages}):"
    )
);

## Delete Images

Checklist to keep an Image:

1. Subject Area Coverage (> 20%)
2. Distinct Botanical Class Features Visible (Flowers that are closed are deleted)
3. Flowers with natural elements shown (Heavy obstructions are deleted)
4. Focus on single flower or a small collection, but never a wide park landscape

In [ ]:
df_main = pd.read_csv(f"{SPLIT_DIR}/train_folds.csv", dtype={'image_path': str, 'fold': int})

In [ ]:
exclude_indices = [
    1801, 2005, 1653, 1287, 2181, 1822, 2418, 2642, 351, 1481,
    1344, 269, 797, 2076, 639, 2846, 2606, 2461, 1949, 349,
    2801, 2851, 2113, 959, 1249, 1774, 2719, 751, 1766, 1758,
    946, 2759, 2196, 892, 2453, 1295, 2035, 2845, 2066, 2448,
    1195, 1561, 2679, 1269, 1678, 413, 1558, 1341, 1318, 1532,
    471, 1978, 753, 2393, 2006, 527, 772, 251, 714, 1173,
    504, 1777, 1064, 1421, 1482, 1611, 2070, 2081, 1362, 1052,
    998, 2532, 529, 2505, 407, 2519, 1698, 902, 2321, 991,
    1074, 2246, 2648, 294, 1382, 682, 2131, 82, 1343, 1219,
    2032, 2368, 1020, 396, 402, 2332, 1788, 2691, 1326, 1865,
    1508, 2710, 2609, 1519, 1200, 704, 1454, 2234, 2680, 735,
    2156, 1765, 2267, 1373, 846
]

df_cleaned = df_main.drop(index=exclude_indices, errors="ignore")

# Reset the index so row numbers are continuous
df_cleaned = df_cleaned.reset_index(drop=True)

# Save the new version
df_cleaned.to_csv(f"{SPLIT_DIR}/train_folds_v2.csv", index=False)

print(f"Original shape: {df_main.shape}")
print(f"Cleaned shape:  {df_cleaned.shape}")

In [ ]:
df_cleaned = pd.read_csv(f"{SPLIT_DIR}/train_folds_v2.csv", dtype={'image_path': str, 'fold': int})

In [ ]:
# Re-create folds
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=30) # History Split: 42
df_cleaned['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(skf.split(df_cleaned, df_cleaned['label'])):
    df_cleaned.loc[val_idx, 'fold'] = fold


In [ ]:
df_cleaned.to_csv(f"{SPLIT_DIR}/train_folds_v3.csv", index=False)

In [ ]:
counts = pd.crosstab(df_cleaned['fold'], df_cleaned['label'], margins=True)
print("=== Raw Counts per Fold ===")
print(counts)

# 2. Inspect percentages per class across all folds (should be nearly identical)
proportions = pd.crosstab(df_cleaned['fold'], df_cleaned['label'], normalize='index') * 100
print("\n=== Class Distribution (%) per Fold ===")
print(proportions.round(2))